# 学习率

上一章，我们用梯度下降法进行了第一次模型训练——方向是对的，但一步把参数从 `[0.5, 0.5]` 更新到了 `[6854, 14147]`，损失值随之爆炸，模型训练失败。

问题的根源在于：**梯度只告诉我们下坡的方向，却不能决定步子应该迈多大。**

梯度很大，只说明当前这个方向的坡度很陡，并不意味着应该迈一大步。步子太大，会直接越过山谷，冲到对面的坡上，反而比出发点更高。这种现象称为**发散**（Divergence）。

---

解决办法是引入一个比例系数，把每次更新的步幅等比例缩小，这个系数称为**学习率**（Learning Rate），记作 $\eta$。

加入学习率后，梯度下降的更新公式变为：

$$
w_{\text{new}} = w_{\text{old}} - \eta \cdot \frac{\partial L}{\partial w}
$$
$$
b_{\text{new}} = b_{\text{old}} - \eta \cdot \frac{\partial L}{\partial b}
$$

学习率需要仔细选择：

* **太大**：更新步幅过大，损失值左右震荡，甚至发散；
* **太小**：更新步幅过小，损失值下降极慢，需要大量训练才能收敛；
* **合适**：损失值稳定地逐步下降，最终收敛到最优解附近。

## 超参数

学习率是我们遇到的第一个**超参数**（Hyperparameter）。

模型参数（权重和偏置）是模型**自动学习**的对象——每次训练后，它们会根据梯度自动更新。

超参数则不同：它们**不会被模型训练自动调整**，而是需要我们在训练开始前根据经验和实验手动设定。学习率控制的是训练步幅，后续章节还会遇到控制数据批次大小、训练轮数等其他超参数。

不断测试、调整超参数以获得最佳训练效果的过程，称为**调参**（Hyperparameter Tuning）。

In [1]:
import numpy as np

In [2]:
class Tensor:

    def __init__(self, data):
        self.data = np.array(data)
        self.grad = np.zeros_like(self.data)
        self.gradient_fn = None
        self.parents = set()

    def backward(self):
        if self.gradient_fn is not None:
            self.gradient_fn()

        for p in self.parents:
            p.backward()

    def __str__(self):
        return f'Tensor({self.data})'

## 数据

In [3]:
feature = Tensor([28.1, 58.0])
label = Tensor([165])

## 模型


In [4]:
class Linear:

    def __init__(self, in_size, out_size):
        self.weight = Tensor(np.ones((out_size, in_size)) / in_size)
        self.bias = Tensor(np.zeros(out_size))

    def __call__(self, x: Tensor):
        return self.forward(x)

    def forward(self, x: Tensor):
        p = Tensor(x.data @ self.weight.data.T + self.bias.data)

        def gradient_fn():
            self.weight.grad += p.grad * x.data
            self.bias.grad += np.sum(p.grad)

        p.gradient_fn = gradient_fn
        return p

    @property
    def parameters(self):
        return [self.weight, self.bias]

## 损失函数（均方误差）

In [5]:
class MSELoss:

    def __call__(self, p: Tensor, y: Tensor):
        return self.loss(p, y)

    def loss(self, p: Tensor, y: Tensor):
        mse = Tensor(np.mean(np.square(y.data - p.data)))

        def gradient_fn():
            p.grad += -2 * (y.data - p.data)

        mse.gradient_fn = gradient_fn
        mse.parents = {p}
        return mse

## 优化器（随机梯度下降）

In [6]:
class SGDOptimizer:

    def __init__(self, parameters, lr):
        self.parameters = parameters
        self.lr = lr

    def step(self):
        for p in self.parameters:
            p.data -= p.grad * self.lr

In [7]:
LEARNING_RATE = 0.00001

In [8]:
layer = Linear(2, 1)
loss_fn = MSELoss()
optimizer = SGDOptimizer(layer.parameters, lr=LEARNING_RATE)

In [9]:
prediction = layer(feature)
loss = loss_fn(prediction, label)
loss.backward()
optimizer.step()

In [10]:
prediction = layer(feature)
loss = loss_fn(prediction, label)
print(f'prediction:\t{prediction}\nloss:\t{loss}')

prediction:	Tensor([53.18309379])
loss:	Tensor(12503.020514375934)
